In [4]:
from pathlib import Path
import os
from epi2hit import *

# ---- base dirs ----
DATA_DIR    = Path("/home/fwm482/epi2hit/data")     
RESULTS_DIR = DATA_DIR / "results"                  
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- input files ----
METH_TSV     = DATA_DIR / "chr8_methylation.tsv"
ATAC_BED     = DATA_DIR / "Corces_PRAD_Peaks_hg19_chr8.bed"
GENE_REF_BED = DATA_DIR / "gencode.v19.protein_coding_gene.minimal_chr.bed.gz"
CHROMHMM_BED = DATA_DIR / "chromHMM_chr8.bed"
LOOPS_BEDPE  = DATA_DIR / "loops.bedpe.gz"
EXPR_TSV     = DATA_DIR / "expression.tsv"          

# ---- output files ----
PROBES_WITH_LOOPS_TSV        = RESULTS_DIR / "probes_with_loops.tsv"
PROBES_WITH_LOOPS_LINREG_TSV = RESULTS_DIR / "probes_with_loops_linreg.tsv"
CORR_PLOTS_DIR               = RESULTS_DIR / "merged_region_corr_plots"
TCGA_DICT_PKL                = RESULTS_DIR / "TCGA_gene_ann_me.pkl"
CHROMHMM_COUNTS_PNG          = RESULTS_DIR / "chromHMM_counts.png"
ECPG_TSV                     = RESULTS_DIR / "eCpGs.tsv"

FOREST_DIR = RESULTS_DIR / "forest_linreg"
PEAKS_DIR  = RESULTS_DIR / "peaks"
FOREST_DIR.mkdir(parents=True, exist_ok=True)
PEAKS_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
# =========================
# 1) Load methylation + ATAC
# =========================
me_full = read_dataset(str(METH_TSV))


# ATAC file
atac= read_dataset(str(ATAC_BED), sep='\t', index_col=None)


# =========================
# 2) Restrict to probes in open chromatin
# =========================
me_open = find_open_chromatin_probes(me_full, atac)


# =========================
# 3) Map probes to nearest genes
# =========================
me_gb = probes2genes(
    me_open,
    str(GENE_REF_BED),
)

# =========================
# 4) Apply stromal/immune/epiCMIT filters
# =========================
me_filtered = prelim_filters(me_gb)  

# =========================
# 5) Add Pomerantz / chromHMM states + save barplot in RESULTS_DIR
# =========================
me_with_pom = pomerantz_state(
    me_filtered,
    reference_bed_path=str(CHROMHMM_BED),
    output_png_path=str(CHROMHMM_COUNTS_PNG),
)

# just in case: make indices unique
me_with_pom = resolve_duplicates(me_with_pom)

# =========================
# 6) Merge correlated probes within regions
# =========================
me_merged = process_probes(
    me_filtered2=me_with_pom,
    correlation_threshold=0.6,
    num_cores=4,
    corr_plots_dir=str(CORR_PLOTS_DIR)
)

# =========================
# 7) Annotate with loops + save full table
# =========================
me_with_loops = annotate_probes_with_loops(
    probe_data=me_merged,
    loop_bedpe_path=str(LOOPS_BEDPE),
    output_path=str(PROBES_WITH_LOOPS_TSV),
)

# =========================
# 8) Build methylation–expression dict + save pickle
# =========================
tcga_dict = prepare_TCGA_dict(
    me_with_loops=me_with_loops,
    exp_path=str(EXPR_TSV),
    output_path=str(TCGA_DICT_PKL),
)

# =========================
# 9) Run linear regression + forest plot in FOREST_DIR
# =========================
res = lin_reg(
    linreg_dict=tcga_dict,
    plot=True,
    plot_dir=str(FOREST_DIR),
)

# =========================
# 10) Keep only probes that have significant linreg hits
# =========================
sliced_probes = slice_probes_with_loops_by_linreg(
    linreg_results=res,
    probes_path=str(PROBES_WITH_LOOPS_TSV),
    output_path=str(PROBES_WITH_LOOPS_LINREG_TSV),
)

# =========================
# 11) Categorize methylation + detect peaks, save plots in PEAKS_DIR
# =========================
labeled_methyl, peak_df = methylation(
    sliced_probes,
    path=False,
    save=True,
    output_dir=str(RESULTS_DIR),
    base_name="methylation",
    plot=True,              
    peaks_dir=str(PEAKS_DIR)
)

# =========================
# 12) Final eCpG selection + save
# =========================
eCpGs = filter_probes_by_methylation(
    labeled_methyl,
    threshold=0.8,
    save=True,
    output_path=str(ECPG_TSV),
)

eCpGs.head()


Merging probes within regions: 100%|██████████| 768/768 [00:34<00:00, 22.49it/s]
Preparing methylation-expression pairs for linreg: 100%|██████████| 3210/3210 [00:02<00:00, 1419.60it/s]
Categorizing methylation, detecting peaks: 100%|██████████| 109/109 [00:32<00:00,  3.38it/s]


,chrom,start,end,TCGA-2A-A8VL-01,TCGA-2A-A8VO-01,TCGA-2A-A8VT-01,TCGA-2A-A8VV-01,TCGA-2A-A8VX-01,TCGA-2A-A8W1-01,TCGA-2A-A8W3-01,...,TCGA-ZG-A9LN-01,TCGA-ZG-A9LS-01,TCGA-ZG-A9LU-01,TCGA-ZG-A9LY-01,TCGA-ZG-A9LZ-01,TCGA-ZG-A9M4-01,TCGA-ZG-A9MC-01,TCGA-ZG-A9N3-01,TCGA-ZG-A9ND-01,TCGA-ZG-A9NI-01
ERICH1_cg00345025_Bival_P,chr8,743875,743876,Medium_methyl,Medium_methyl,Medium_methyl,Not/Low_methyl,Highly_methyl,Medium_methyl,Medium_methyl,...,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Not/Low_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Highly_methyl,Medium_methyl
DLGAP2_cg26942892_Primed_PSE,chr8,1581568,1581569,Highly_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Highly_methyl,Medium_methyl,Highly_methyl,...,NaN,Medium_methyl,Medium_methyl,Medium_methyl,Not/Low_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Highly_methyl,Highly_methyl
Ls_ERICH1_cg24578875_Act_PSP,chr8,1728703,1728704,Highly_methyl,Highly_methyl,Medium_methyl,Medium_methyl,Highly_methyl,Not/Low_methyl,Medium_methyl,...,Highly_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Highly_methyl,Medium_methyl,Medium_methyl
Ls_CLN8_cg17290127_Act_PSP,chr8,1728777,1728778,Highly_methyl,Medium_methyl,Highly_methyl,Medium_methyl,Medium_methyl,Not/Low_methyl,Highly_methyl,...,Highly_methyl,Highly_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Highly_methyl,Medium_methyl,Medium_methyl
ARHGEF10_cg10434754_Primed_nPSE,chr8,1785833,1785834,Medium_methyl,Medium_methyl,Medium_methyl,Not/Low_methyl,Medium_methyl,Medium_methyl,Highly_methyl,...,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Not/Low_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Medium_methyl,Highly_methyl
